# Evaluating a Voice Agent

*Does it sound right read aloud?*

The voice notebook makes a promise twice. `RESEARCH_INSTRUCTIONS` asks for "3-5 sentences of plain,
conversational language. No markdown, no bullet lists, no URLs." `VOICE_INSTRUCTIONS` asks the model
to keep replies "short and natural &mdash; you are being heard, not read." Nothing checked either one.

That is a quiet failure. A report that reverts to bullet points and bare URLs still *works* &mdash; the
pipeline runs, the tool returns, the model narrates. It just sounds terrible, and the only way you
find out is by listening to it. That is exactly the kind of regression an eval should catch first.

This notebook grades one thing: **spoken-friendliness**. Not routing, not grading whether the facts
are right &mdash; those are different metrics with different rubrics. Just: would this work as speech?

It does it in two layers, because the failure has two halves:

| | Grades | Judge | Catches |
| --- | --- | --- | --- |
| **Offline** | the deep agent's report text | gateway `ChatOpenAI` | markdown and URLs *at the source* |
| **Online** | live traced conversations | Gemini, listening to the WAV | truncation, talk-over, dead air |

The offline layer is the regression gate: deterministic, no microphone, runs in CI. The online layer
catches what replay cannot.

In [ ]:
%load_ext autoreload
%autoreload 2

# Setup

The judges live in `util/voice_evals.py`, and the agent under test comes from `util/research.py`
&mdash; the *same* module the voice notebook imports. That sharing is the point: if this notebook
re-declared `RESEARCH_INSTRUCTIONS`, the two copies would drift the first time either was tuned, and
the eval would go on happily scoring a prompt you no longer ship.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

from langsmith import Client

from util import (
    SPOKEN_CHECKS,
    build_research_agent,
    build_research_model,
    grade_report,
    make_spoken_judge,
    stream_report,
)

client = Client()

# The agent under test: the same coordinator + `researcher` subagent the voice notebook runs.
research_agent = build_research_agent()

# The judge. Reference-free — it reads the report and asks "would this work as speech?", so
# no golden answer is needed. temperature is left at the gateway default; the rubric is
# strict enough that the checks are stable.
judge = make_spoken_judge(build_research_model())

# A dataset that *baits* the failure

An eval is only as good as its examples, and "write a short spoken answer" is easy to pass on an easy
question. So none of these are easy questions. Each one is phrased the way somebody actually speaks,
and each is deliberately shaped to tempt the model into a different kind of unspeakable output:

- a **comparison** &mdash; the natural formatting instinct is a table
- a **multi-part question** &mdash; tempts a numbered list, one item per part
- a **"what's the current X"** question &mdash; tempts a citation or a bare URL
- a **taxonomy question** &mdash; tempts bullets, one per category

If the agent holds the line on all four, the prompt is doing its job. The `outputs` field records
which trap each example is setting; the judge never reads it, but it is what you want to see when a
score drops and you are trying to work out which instinct broke through.

In [ ]:
DATASET_NAME = "deepagents-voice-spoken-friendliness-v1"

examples = [
    {
        "inputs": {"topic": "Which is cheaper to run over five years, a heat pump or a gas boiler?"},
        "outputs": {"baits": "table"},
    },
    {
        "inputs": {"topic": (
            "Tell me about the James Webb telescope — when did it launch, what has it actually "
            "found, and what is next for it?"
        )},
        "outputs": {"baits": "numbered list"},
    },
    {
        "inputs": {"topic": "What is the Bank of England base rate right now, and when is it reviewed next?"},
        "outputs": {"baits": "citation or bare URL"},
    },
    {
        "inputs": {"topic": "What are the main differences between the types of COVID vaccine?"},
        "outputs": {"baits": "bullet list"},
    },
]

if client.has_dataset(dataset_name=DATASET_NAME):
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
else:
    dataset = client.create_dataset(
        DATASET_NAME,
        description="Spoken-friendliness evals for the voice agent's research reports.",
    )

# Same guard as deepagents-evals.ipynb: never silently overwrite remote examples. If they
# have diverged, that is a new dataset version, not an edit — experiments already scored
# against the old examples would otherwise become uncomparable without any signal.
existing_examples = list(client.list_examples(dataset_id=dataset.id))
if not existing_examples:
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created {len(examples)} examples.")
else:
    remote_examples = [
        {"inputs": example.inputs, "outputs": example.outputs} for example in existing_examples
    ]
    by_topic = lambda example: example["inputs"]["topic"]
    if sorted(remote_examples, key=by_topic) != sorted(examples, key=by_topic):
        raise RuntimeError(
            f"{DATASET_NAME!r} already exists with different examples. "
            "Use a new versioned dataset name rather than overwriting remote data."
        )
    print(f"Reusing {DATASET_NAME!r} with {len(existing_examples)} examples.")

print(f"Dataset: {dataset.url}")

# The rubric

Four independent booleans, all of which must earn `true`. They live in `util/voice_evals.py` as a
`TypedDict` of `Annotated` fields, which becomes the judge's structured output &mdash; the same shape
`deepagents-evals.ipynb` uses for its itinerary grader.

| check | fails when |
| --- | --- |
| `plain_sentences` | not 3&ndash;5 conversational sentences (a list with the bullets stripped off is still a list) |
| `no_markup` | any markdown at all &mdash; bold, headers, bullets, code fences, tables |
| `speakable` | contains something a listener cannot hear: a URL, a bare domain, a `[1]` citation, an emoji |
| `leads_with_answer` | opens with preamble or restates the question instead of answering it |

Scoring is deliberately **strict** &mdash; `int(all(...))`, no partial credit. A report with one bare
URL in it is not 75% speakable; it is a report that will make the assistant read a URL out loud. The
per-check breakdown still lands in the feedback comment, so a failure tells you *which* instinct broke
through without the score pretending the answer was nearly fine.

# Test the judge before spending experiment calls

The judge is the measuring instrument, so check it against two reports where you already know the
answer. This costs two LLM calls and catches a miscalibrated rubric before it silently mis-scores a
whole experiment.

In [ ]:
GOOD_REPORT = (
    "Heat pumps usually work out cheaper over five years, though the gap depends heavily on what you "
    "pay for electricity. Running costs are roughly a third lower than a gas boiler in most homes, "
    "because a heat pump moves heat rather than burning fuel. The catch is the install cost, which is "
    "several times higher, so the payback period is what really decides it for a given household."
)

BAD_REPORT = """Here's the comparison:

## Five-year running costs

| System | Annual cost | 5-year total |
| --- | --- | --- |
| **Heat pump** | £850 | £4,250 |
| **Gas boiler** | £1,200 | £6,000 |

Full methodology: https://www.energysavingtrust.org.uk/heat-pump-costs [1]"""

for label, report in (("GOOD", GOOD_REPORT), ("BAD", BAD_REPORT)):
    result = grade_report(judge, report, "Which is cheaper, a heat pump or a gas boiler?")
    print(f"{label}: score={result['score']}")
    print(f"  {result['comment']}\n")

# The evaluator, and the run

`stream_report` is the same function the voice loop calls &mdash; its `panel` argument is optional, so
outside the notebook UI it just returns the final report text. That means the thing being graded here
is literally the thing that gets spoken, with no re-implementation in between.

It is `async`, so this uses **`client.aevaluate`** rather than `client.evaluate`.

In [ ]:
async def spoken_report(inputs: dict) -> dict:
    """The target: run the deep agent and return the text destined to be read aloud."""
    return {"report": await stream_report(research_agent, inputs["topic"])}


def spoken_friendliness(inputs: dict, outputs: dict) -> dict:
    """Reference-free evaluator: would this text work as speech?"""
    return grade_report(judge, outputs.get("report", ""), inputs["topic"])


results = await client.aevaluate(
    spoken_report,
    data=DATASET_NAME,
    evaluators=[spoken_friendliness],
    experiment_prefix="voice-spoken-friendliness",
    max_concurrency=2,
)

print(f"Experiment: {results.experiment_name}")
print(f"View results: {results.url}")

# Layer two: what the offline eval cannot see

Everything above grades **text**. But what the caller actually hears is Gemini Live *narrating* that
report &mdash; and narration has its own failure modes that no transcript records:

- a reply cut off mid-sentence because the socket hiccuped
- the assistant talking over the user instead of yielding on a barge-in
- a robotic list cadence, even when the words themselves are fine

So the offline layer catches the *upstream cause* (the deep agent emitted markdown) while the online
layer catches the delivery. You need both; neither substitutes for the other.

This is possible at all because the voice notebook now wraps its session in `wrap_gemini_live`, which
attaches the conversation to the trace as a stereo WAV &mdash; **you on the left channel, the assistant
on the right, as actually played**. Audio dropped by a barge-in never reaches the file, so the
recording is what happened in the room rather than what the model generated.

`util/voice_evals.py` has a second rubric for this, `DeliveryGrade`, scored by a Gemini flash judge.
Gemini specifically: LangSmith's docs note audio attachments are only supported by Gemini judges, and
it pins the endpoint at Google rather than the gateway &mdash; which answers a bare `403 Forbidden`,
the same trap that breaks the Live handshake.

**The rubric leans hard on the channels**, so the prompt has to defend them. On the first run the judge
read the assistant's greeting on the right channel and reported it as the *user* greeting the assistant,
then failed the clip for never replying. The prompt now states that attribution follows the channel and
never the content, and that a quiet left channel is expected rather than a fault &mdash; the microphone is
muted while the assistant speaks. Worth re-reading the judge's reasoning on your own recordings to
confirm it is really hearing two channels and not a downmix.

In [ ]:
from util import DELIVERY_CHECKS, build_audio_judge, grade_delivery

# Grade the most recent traced conversation. Hold a real voice session in
# deepagents-voice-no-tavily.ipynb first, or point PROJECT at wherever those traces land.
PROJECT = os.getenv("LANGSMITH_PROJECT")

# A flash model: the rubric is four yes/no judgements about how a clip sounded, which
# does not need a pro-tier model, and audio tokens are not cheap enough to waste.
AUDIO_JUDGE_MODEL = "gemini-3.8-flash"

conversations = list(client.list_runs(
    project_name=PROJECT, filter='eq(name, "realtime_session")', is_root=True, limit=1,
))

if not conversations:
    print(f"No realtime_session traces in {PROJECT!r} yet — hold a voice session first.")
else:
    run = client.read_run(conversations[0].id)
    attachment = (run.attachments or {}).get("conversation")
    if attachment is None:
        print("That conversation has no audio attached.")
    else:
        judge_audio = build_audio_judge(AUDIO_JUDGE_MODEL)
        result = grade_delivery(judge_audio, attachment["reader"].read())
        print(f"delivery_quality: score={result['score']}")
        print(f"  {result['comment']}")

# Putting the audio judge on live traces

Running it by hand above proves the rubric works. To have it score every conversation, attach it as an
**online evaluator** in the LangSmith UI &mdash; mapping an attachment into an evaluator prompt uses the
`{{attachments}}` template variable in the **Template variables** editor, which has no SDK equivalent.

In your tracing project &rarr; **Evaluators** &rarr; **+ Evaluator** (there is a **Voice Evaluation**
template category worth starting from):

1. Paste `DELIVERY_JUDGE_PROMPT` from `util/voice_evals.py` as the prompt, and recreate the four
   `DeliveryGrade` booleans as the feedback configuration.
2. Map `{{attachment.conversation}}` so the judge receives the WAV.
3. Pick a Gemini model.
4. Filter to the conversation root, not the event spans: `eq(name, "realtime_session")` and
   `is_root = true`. Without this you will score every transcript fragment separately.

Three settings worth choosing deliberately rather than accepting:

- **Sampling rate.** Audio judging is not cheap. Start around `0.2`.
- **Retention.** Online evaluators auto-upgrade the traces they score to extended retention, which
  changes what those traces cost. Opt out if you do not want that.
- **Spend limit.** Set a weekly per-evaluator cap under **Advanced**.

# What to remember

- **Grade the thing you ship.** The prompts live in `util/research.py` precisely so the eval and the
  voice loop cannot drift apart. A duplicated prompt is an eval that slowly stops meaning anything.
- **Bait the failure.** Four questions, each engineered to tempt a different kind of unspeakable
  output. An eval built from easy questions passes right up until production.
- **Be strict where the failure is binary.** One bare URL is not a partial pass &mdash; the assistant
  either reads a URL aloud or it does not. The per-check comment carries the nuance instead.
- **Text and audio catch different things.** Offline grades the report and runs in CI; online grades
  the delivery and needs a real conversation. Neither one subsumes the other.